## Exploração inicial — Acessos de Banda Larga Fixa (Anatel)

Objetivo: conhecer a base de acessos de banda larga fixa (SCM) da Anatel antes de construir o pipeline de dados. Aqui confirmo formato, colunas, volume e as decisões que orientam o ETL.

**Fonte:** [dados abertos da Anatel](https://www.anatel.gov.br/dadosabertos/paineis_de_dados/acessos/acessos_banda_larga_fixa.zip), baixados com `src/extract.py`.

### 1. Arquivos disponíveis

O zip tem cerca de 1 GB e, descompactado, ocupa uns 10 GB em 26 arquivos CSV:

- `Acessos_Banda_Larga_Fixa_AAAA.csv`: a base principal, com uma linha por mês, empresa, município e característica do acesso. **É a base usada no projeto.**
- `..._Colunas.csv`: os mesmos dados em formato largo, com um mês por coluna. Não será usado.
- `Densidade_Banda_Larga_Fixa.csv`: acessos por 100 habitantes, por município. Pode virar uma análise complementar.

Atenção: a lista é ordenada alfabeticamente, então os arquivos são escolhidos **pelo nome**, nunca pela posição.

In [2]:
#1
from pathlib import Path
import pandas as pd 

pd.set_option("display.max_columns", None)

#Funciona tanto rodar na raíz como na pasta notebooks/
ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
PASTA = ROOT / "data" / "raw" / "acessos_banda_larga_fixa"

arquivos = sorted(PASTA.rglob("*.csv"))
for a in arquivos:
    print(f"{a.name:55} {a.stat().st_size / 1e6:8.1f} MB")

Acessos_Banda_Larga_Fixa_2007-2010.csv                      97.8 MB
Acessos_Banda_Larga_Fixa_2007-2010_Colunas.csv              11.6 MB
Acessos_Banda_Larga_Fixa_2011-2012.csv                     190.8 MB
Acessos_Banda_Larga_Fixa_2011-2012_Colunas.csv              14.6 MB
Acessos_Banda_Larga_Fixa_2013-2014.csv                     237.0 MB
Acessos_Banda_Larga_Fixa_2013-2014_Colunas.csv              20.5 MB
Acessos_Banda_Larga_Fixa_2015-2016.csv                     276.2 MB
Acessos_Banda_Larga_Fixa_2015-2016_Colunas.csv              21.8 MB
Acessos_Banda_Larga_Fixa_2017-2018.csv                     344.1 MB
Acessos_Banda_Larga_Fixa_2017-2018_Colunas.csv              29.4 MB
Acessos_Banda_Larga_Fixa_2019-2020.csv                     428.8 MB
Acessos_Banda_Larga_Fixa_2019_2020_Colunas.csv              36.3 MB
Acessos_Banda_Larga_Fixa_2021.csv                          908.9 MB
Acessos_Banda_Larga_Fixa_2021_Colunas.csv                  168.0 MB
Acessos_Banda_Larga_Fixa_2022.csv               

### 2. O arquivo cru

Antes de usar o pandas, leio as primeiras linhas como texto para confirmar o formato com os próprios olhos:

- separador `;`
- encoding UTF-8, com os acentos corretos
- números decimais com **vírgula** (padrão brasileiro), como em `10,000000`

In [3]:
#2
arquivo = PASTA / "Acessos_Banda_Larga_Fixa_2026.csv"
print(arquivo.name)
with open(arquivo, encoding="utf-8") as f:
    for _ in range(3):
        print(f.readline())

Acessos_Banda_Larga_Fixa_2026.csv
﻿Ano;Mês;Grupo Econômico;Empresa;CNPJ;Porte da Prestadora;UF;Município;Código IBGE Município;Faixa de Velocidade;Velocidade;Tecnologia;Meio de Acesso;Tipo de Pessoa;Tipo de Produto;Acessos

2026;9;HUGHES;HUGHES;05206385000161;Pequeno Porte;MG;Itueta;3134103;2Mbps a 12Mbps;10,000000;VSAT;Satélite;Pessoa Física;INTERNET;3

2026;9;TELEFONICA;VIVO;02558157000162;Grande Porte;SP;Cosmópolis;3512803;512kbps a 2Mbps;1,000000;ADSL2;Cabo Metálico;Pessoa Física;INTERNET;4



### 3. Colunas e tipos

A base tem **16 colunas**: período (Ano, Mês), prestadora (Grupo Econômico, Empresa, CNPJ, Porte), localização (UF, Município, Código IBGE), características do acesso (Faixa de Velocidade, Velocidade, Tecnologia, Meio de Acesso, Tipo de Pessoa, Tipo de Produto) e a medida, **Acessos**.

**Cuidado importante:** `CNPJ` e `Código IBGE Município` precisam ser lidos como **texto**. Muitos CNPJs começam com zero e, lidos como número, perderiam esse zero, quebrando filtros e junções.

In [4]:
#3
amostra = pd.read_csv(arquivo, sep=";", nrows=5000, dtype=str)
print(f"Colunas da tabela: {list(amostra.columns)}")
amostra.head()

Colunas da tabela: ['Ano', 'Mês', 'Grupo Econômico', 'Empresa', 'CNPJ', 'Porte da Prestadora', 'UF', 'Município', 'Código IBGE Município', 'Faixa de Velocidade', 'Velocidade', 'Tecnologia', 'Meio de Acesso', 'Tipo de Pessoa', 'Tipo de Produto', 'Acessos']


,Ano,Mês,Grupo Econômico,Empresa,CNPJ,Porte da Prestadora,UF,Município,Código IBGE Município,Faixa de Velocidade,Velocidade,Tecnologia,Meio de Acesso,Tipo de Pessoa,Tipo de Produto,Acessos
0,2026,9,HUGHES,HUGHES,05206385000161,Pequeno Porte,MG,Itueta,3134103,2Mbps a 12Mbps,"10,000000",VSAT,Satélite,Pessoa Física,INTERNET,3
1,2026,9,TELEFONICA,VIVO,02558157000162,Grande Porte,SP,Cosmópolis,3512803,512kbps a 2Mbps,"1,000000",ADSL2,Cabo Metálico,Pessoa Física,INTERNET,4
2,2026,9,OUTROS,PB NET,05658533000189,Pequeno Porte,PB,Cuité,2505105,12Mbps a 34Mbps,"20,000000",Wi-Fi,Rádio,Pessoa Física,INTERNET,16
3,2026,9,OUTROS,G3 Telecom,13133062000113,Pequeno Porte,PI,Valença do Piauí,2211308,> 34Mbps,"600,000000",FTTH,Fibra,Pessoa Física,INTERNET,18
4,2026,9,OUTROS,WKVE TELECOM,00989304000123,Pequeno Porte,MG,Governador Valadares,3127701,> 34Mbps,"200,000000",FTTH,Fibra,Pessoa Física,INTERNET,1


### 4. Filtrando Alagoas sem estourar a memória

Só o arquivo de 2026 tem quase 1 GB. Em vez de carregá-lo inteiro, leio em pedaços de 500 mil linhas (`chunksize`) e guardo apenas as linhas de Alagoas.

**Resultado (2026):** 6.355.655 linhas no Brasil, das quais **69.300 são de Alagoas** (cerca de 1%). Essa mesma técnica será usada no `src/transform.py`.

In [5]:
#4
pedacos = []
total_linhas = 0
for chunk in pd.read_csv(arquivo, sep=";", chunksize=500_000, dtype={"CNPJ": str, "Código IBGE Município":str}):
    total_linhas +=len(chunk)
    pedacos.append(chunk[chunk["UF"] == "AL"])

al = pd.concat(pedacos, ignore_index = True)
print(f"Linhas no arquivo: {total_linhas:,} | Linhas de Alagoas: {len(al):,}")

Linhas no arquivo: 6,355,655 | Linhas de Alagoas: 69,300


### 5. Perfil dos dados de Alagoas

- **Período em 2026:** janeiro a setembro
- **Municípios:** 102, todos os do estado
- **Empresas reportando:** 379, um retrato da fragmentação do mercado, com centenas de provedores regionais
- **Tecnologia:** FTTH (fibra até a casa) é a mais frequente

**Regra fundamental:** a coluna `Acessos` é um **estoque mensal**, ou seja, quantos acessos existiam naquele mês. **Nunca se soma Acessos entre meses diferentes**; compara-se um mês com outro.

In [6]:
#5
for coluna in ["Ano", "Mês", "Tecnologia", "Meio de Acesso",
               "Tipo de Pessoa", "Tipo de Produto", "Porte da Prestadora"]:
    if coluna in al.columns:
        print(f"\n--- {coluna} ---")
        print(al[coluna].value_counts().head(15))

print("\nMunicípios:", al["Município"].nunique())   # Alagoas tem 102
print("Empresas:", al["Empresa"].nunique())


--- Ano ---
Ano
2026    69300
Name: count, dtype: int64

--- Mês ---
Mês
7    8103
5    8078
6    8033
8    7957
3    7750
4    7595
2    7513
1    7189
9    7082
Name: count, dtype: int64

--- Tecnologia ---
Tecnologia
FTTH           42105
VSAT            7790
ETHERNET        5676
Wi-Fi           5137
ADSL2           1787
HDSL            1723
FTTB            1374
ADSL1           1280
HFC              970
DWDM             630
TDMA             330
Cable Modem      182
FWA              172
FR                82
OFDMA/TDD         53
Name: count, dtype: int64

--- Meio de Acesso ---
Meio de Acesso
Fibra            48409
Satélite          8129
Cabo Metálico     6322
Rádio             5405
Cabo Coaxial      1035
Name: count, dtype: int64

--- Tipo de Pessoa ---
Tipo de Pessoa
Pessoa Física      35460
Pessoa Jurídica    33840
Name: count, dtype: int64

--- Tipo de Produto ---
Tipo de Produto
INTERNET          59146
LINHA_DEDICADA     6269
M2M                3315
OUTROS              570
Name: 

### 6. Quem domina o mercado (setembro de 2026)

Alagoas tinha **387.165 acessos** de banda larga fixa em setembro de 2026.

- A **Brisanet** lidera com cerca de 117 mil acessos (~30%), seguida da **Claro** (~22%).
- Abaixo delas há uma longa cauda de provedores regionais.
- A **Starlink** (satélite) já aparece entre as 15 maiores. Isso sugere demanda em áreas onde a fibra ainda não chegou e merece uma análise própria.

Observação: este ranking soma todos os tipos de produto. Nas análises de mercado do projeto, o filtro será `Tipo de Produto = INTERNET`.

In [7]:
#6
setembro = al[al["Mês"] == al["Mês"].max()]
print("Total de acessos em AL (set/2026):", f"{setembro['Acessos'].sum():,}")
(setembro.groupby(["Grupo Econômico", "Empresa"])["Acessos"]
         .sum().sort_values(ascending=False).head(15))

Total de acessos em AL (set/2026): 387,165


Grupo Econômico   Empresa                                   
BRISANET          BRISANET                                      116856
TELECOM AMERICAS  CLARO                                          86617
TELEFONICA        VIVO                                           21543
OUTROS            Dtel                                           13356
OI                OI                                              5991
OUTROS            TELECOM INTERNET                                5788
                  G S Net                                         3850
                  REELUNET                                        3679
                  R E COMERCIO E SERVICOS DE INTERNET LTDA        3480
                  SPEEDY TELECOM                                  3353
                  Inova +                                         2928
                  STARLINK BRAZIL SERVICOS DE INTERNET LTDA.      2859
                  MARBRIELE                                       2831
                

### 7. Mudança de metodologia em 2021

Os arquivos até 2020 têm **13 colunas**; a partir de 2021, **16**. Em 2021 a Anatel mudou a coleta: passou a separar meio de acesso e tecnologia e trocou as faixas de velocidade pelas velocidades contratadas.

Comparar dados de antes e depois dessa mudança geraria conclusões erradas, então o projeto usa apenas **janeiro de 2021 a setembro de 2026**.

In [8]:
#7
for a in sorted(PASTA.glob("Acessos_Banda_Larga_Fixa_*.csv")):
    if "Colunas" in a.name or "Total" in a.name:
        continue
    cols = pd.read_csv(a, sep=";", nrows=1).columns.tolist()
    print(f"{a.name:45} {len(cols)} colunas")

Acessos_Banda_Larga_Fixa_2007-2010.csv        13 colunas
Acessos_Banda_Larga_Fixa_2011-2012.csv        13 colunas
Acessos_Banda_Larga_Fixa_2013-2014.csv        13 colunas
Acessos_Banda_Larga_Fixa_2015-2016.csv        13 colunas
Acessos_Banda_Larga_Fixa_2017-2018.csv        13 colunas
Acessos_Banda_Larga_Fixa_2019-2020.csv        13 colunas
Acessos_Banda_Larga_Fixa_2021.csv             16 colunas
Acessos_Banda_Larga_Fixa_2022.csv             16 colunas
Acessos_Banda_Larga_Fixa_2023.csv             16 colunas
Acessos_Banda_Larga_Fixa_2024.csv             16 colunas
Acessos_Banda_Larga_Fixa_2025.csv             16 colunas
Acessos_Banda_Larga_Fixa_2026.csv             16 colunas


### Conclusões para o pipeline

1. Usar os arquivos anuais de **2021 a 2026** (16 colunas, metodologia consistente).
2. Ler em pedaços (`chunksize`) e filtrar **UF = AL**.
3. Ler `CNPJ` e `Código IBGE Município` como texto, e `Velocidade` com `decimal=","`.
4. Converter Ano e Mês numa data (`ano_mes`, sempre o dia 1) para facilitar as séries temporais.
5. Nas análises de mercado, filtrar `Tipo de Produto = INTERNET`.
6. Nunca somar `Acessos` entre meses diferentes.
7. Limitação: algumas prestadoras reportam de forma intermitente (subnotificação), o que deve ser registrado no README.